In [ ]:
import pandas as pd
data='/kaggle/input/misogony-classify/final_labels.csv'

In [ ]:
df = pd.read_csv(data)

In [ ]:
df = df.rename(columns={"level_3": "label"})

In [ ]:
df['label'] = df['label'].apply(lambda x: 1 if x == "Misogynistic" else 0)

In [ ]:
df_train = df[df['split']=='train']
df_test = df[df['split']=='test']

In [ ]:
df_train_itc=df_train[df_train['strength']=='Nature of the abuse is Implicit']
df_train_etc=df_train[df_train['strength']=='Nature of the abuse is Explicit']

In [ ]:
len(df_train_itc)

In [ ]:
def get_qc_examples(df):
    """Creates examples for the training and dev sets."""
    text_and_labels = list(zip(df['body'], df['label']))
    return text_and_labels[1:]

In [ ]:
train_data = get_qc_examples(df_train)
test_data = get_qc_examples(df_test)

In [ ]:
len(train_data)

In [ ]:
len(test_data)

In [ ]:
X=[_ for _,label in train_data]
y=[label for _,label in train_data ]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.9, random_state=42, stratify=y)

In [ ]:
train_data = []
for i in range(len(X_train)):
    train_data.append((X_train[i],y_train[i]))

In [ ]:
from collections import Counter
train_counts = Counter(label for _, label in train_data)
print("Labeled Examples Count:", train_counts)
# Count labels in test examples
test_counts = Counter(label for _, label in test_data)
print("Test Examples Count:", test_counts)

In [ ]:
import torch
import io
import torch.nn.functional as F
import torch.nn as nn
import random
import numpy as np
import time
import math
import pandas as pd
import datetime
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [ ]:
import transformers
from transformers import *

In [ ]:
print(transformers.__version__)

In [ ]:
seed_val = 123
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seed_val)

In [ ]:
max_seq_length = 128
batch_size = 16
out_dropout_rate = 0.5
num_train_epochs = 15
multi_gpu = True
# Scheduler
apply_balance=False
apply_scheduler = False
warmup_proportion = 0.1
# Print
num_labels=2
print_each_n_step = 100
model_name = "FacebookAI/roberta-large"

In [ ]:
if torch.cuda.is_available():    
    # Tell PyTorch to use the GPU.    
    device = torch.device("cuda")
    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    print('We will use the GPU:', torch.cuda.get_device_name(0))
# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
transformer = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(transformer,tokenizer)

In [ ]:
def generate_data_loader(examples, do_shuffle = False, balance_label_examples = False):
  '''
  Generate a Dataloader given the input examples, eventually masked if they are 
  to be considered NOT labeled.
  '''

  input_ids = []
  input_mask_array = []
  label_id_array = []

  # Tokenization 
  for (text, label) in examples:
    encoded_sent = tokenizer.encode(str(text), add_special_tokens=True, max_length=max_seq_length, padding="max_length", truncation=True)
    input_ids.append(encoded_sent)
    label_id_array.append(int(label))
  
  # Attention to token (to ignore padded input wordpieces)
  for sent in input_ids:
    att_mask = [int(token_id > 0) for token_id in sent]                          
    input_mask_array.append(att_mask)
  # Convertion to Tensor
  input_ids = torch.tensor(input_ids) 
  input_mask_array = torch.tensor(input_mask_array)
  label_id_array = torch.tensor(label_id_array, dtype=torch.long)

  # Building the TensorDataset
  dataset = TensorDataset(input_ids, input_mask_array, label_id_array)

  if do_shuffle:
    sampler = RandomSampler
  else:
    sampler = SequentialSampler

  # Building the DataLoader
  return DataLoader(
              dataset,  # The training samples.
              sampler = sampler(dataset), 
              batch_size = batch_size) # Trains with this batch size.

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
train_dataloader=generate_data_loader(train_data,do_shuffle = True,balance_label_examples = apply_balance)
test_dataloader=generate_data_loader(test_data,do_shuffle = True,balance_label_examples = apply_balance)

In [ ]:
for batch in train_dataloader:
    input_ids, attention_mask, label  = batch
    # print(batch)
    print(input_ids)
    print(attention_mask)
    print(label)
    # print(label_mask)
    break

In [ ]:
class BertSentimentClassifier(nn.Module):
    def __init__(self, transformer , num_classes, dropout=0.2):
        super(BertSentimentClassifier, self).__init__()
        self.bert = transformer
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)
        self.softmax=nn.Softmax(dim=-1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [CLS] token representation
        pooled_output = self.dropout(pooled_output)
        logits = self.fc(pooled_output)
        logprobs=self.softmax(logits)
        return logits,logprobs

In [ ]:
config = AutoConfig.from_pretrained(model_name)
hidden_size = int(config.hidden_size)
model = BertSentimentClassifier(transformer, num_classes=2,dropout=0.5)
if torch.cuda.is_available():    
  transformer.cuda()
  if multi_gpu:
    transformer = torch.nn.DataParallel(transformer)

# print(config)

In [ ]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-6)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
c_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = c_p
print("total parameters:",total_params)

In [ ]:
import time
import datetime
training_stats=[]
# Format time function
def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

print_each_n_step = 100  # Frequency of step logging
best_test_accuracy = 0  # Track the best test accuracy

best_f1 = 0.0
best_model_path = "best_classifier.pt"


for epoch in range(num_train_epochs):
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch + 1, num_train_epochs))
    print('Training...')
    model.train()
    epoch_loss = 0
    correct_predictions = 0
    total_samples = 0
    t0 = time.time()

    for step, batch in enumerate(train_dataloader):
        if step % print_each_n_step == 0 and step != 0:
            elapsed = format_time(time.time() - t0)
            print(f"  Batch {step:>5} of {len(train_dataloader)}. Elapsed: {elapsed}.")

        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)


        # Forward Pass
        optimizer.zero_grad()
        outputs,logprobs = model(input_ids, attention_mask)
        loss = criterion(logprobs, labels)
        
        # Backward Pass and Optimization
        loss.backward()
        optimizer.step()

        # Accumulate Loss and Accuracy
        epoch_loss += loss.item()
        predictions = torch.argmax(logprobs, dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = epoch_loss / len(train_dataloader)
    accuracy = correct_predictions / total_samples
    print(f"Training Accuracy: {accuracy:.4f}")
    print(f"Training epoch took: {format_time(time.time() - t0)}")

    # Evaluation
    print("\nRunning Test...")
    model.eval()
    test_loss = 0
    correct_predictions = 0
    total_samples = 0
    total_test_accuracy = 0
   
    total_test_loss = 0
    nb_test_steps = 0
    wrong_predictions=[]
    all_preds = []
    all_labels_ids = []
    for batch in test_dataloader:
        # input_text=batch['text']
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)
        with torch.no_grad():
            outputs,probs= model(input_ids, attention_mask)
            loss = criterion(probs, labels)
            total_test_loss+=loss

        _, preds = torch.max(probs, 1)
        all_preds += preds.detach().cpu()
        all_labels_ids += labels.detach().cpu()

        # for text, pred, true_label in zip(input_text, all_preds, all_labels_ids):
        #     if pred != true_label:
        #         wrong_predictions.append((text, true_label, pred))
        
    all_preds = torch.stack(all_preds).numpy()   
    all_labels_ids = torch.stack(all_labels_ids).numpy()
    test_accuracy = np.sum(all_preds == all_labels_ids) / len(all_preds)
    print("  Accuracy: {0:.3f}".format(test_accuracy))

    macro_f1 = f1_score(all_labels_ids, all_preds, average='macro')
    print("F1 Score: {:.3f}".format(macro_f1))
    # Save best classifier based on F1 score
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"\n✅ Saved new best model at Epoch {epoch + 1} with F1: {best_f1:.4f}")


    # Calculate the average loss over all of the batches.
    avg_test_loss = total_test_loss / len(test_dataloader)
    avg_test_loss = avg_test_loss.item()
    
    # Measure how long the validation run took.
    test_time = format_time(time.time() - t0)
    
    print("  val Loss: {0:.3f}".format(avg_test_loss))
    print("  val took: {:}".format(test_time))
    # print("\nMisclassified Sentences:")
    # for i, (text, true_label, pred) in enumerate(wrong_predictions[:10]):  # Show first 10 errors
    #     print(f"{i+1}. Text: {text}")
    #     print(f"   True Label: {true_label}, Predicted: {pred}\n")
    training_stats.append(
        {
            'epoch': epoch + 1,
            'Training Loss Classifer': avg_loss,
            'Valid. Loss': avg_test_loss,
            'Valid. Accur.': test_accuracy,
            'F1 Score': macro_f1,
            'misclassified':wrong_predictions,
            # 'Training Time': training_time,
            'Test Time': test_time
        }
    )
   

In [ ]:
import matplotlib.pyplot as plt
# training_stats=training_stats[0:8]
# Extract accuracy and F1 score from training_stats
epochs = [entry['epoch'] for entry in training_stats]  # Get epochs
accuracy = [entry['Valid. Accur.'] for entry in training_stats]  # Get accuracy values
f1_scores = [entry['F1 Score'] for entry in training_stats]  # Get F1 scores (store in 'F1 Score' while logging)
test_loss=[entry['Valid. Loss'] for entry in training_stats]
# d_loss=[entry['Training Loss discriminator'] for entry in training_stats]
# g_loss=[entry['Training Loss generator'] for entry in training_stats]
c_loss=[entry['Training Loss Classifer'] for entry in training_stats]
# Plot accuracy and F1 score
plt.figure(figsize=(10, 6))
plt.plot(epochs, accuracy, label="Accuracy", marker="o")
plt.plot(epochs, f1_scores, label="F1 Score", marker="o")
# plt.plot(epochs,test_loss,label='tes_loss',marker="o")
# Adding titles and labels
plt.title("Accuracy and F1 Score over Epochs", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Metrics", fontsize=12)
plt.xticks(epochs)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
plt.plot(epochs, test_loss, label="test_loss", marker="o")
# plt.plot(epochs,d_loss , label="d_loss", marker="x")
# plt.plot(epochs,g_loss , label="g_loss", marker="x")
plt.plot(epochs,c_loss , label="train_loss", marker="x")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Metrics", fontsize=12)
plt.xticks(epochs)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
from lime.lime_text import LimeTextExplainer
import torch.nn.functional as F

model.load_state_dict(torch.load("best_classifier.pt", weights_only=True))
model.eval()


def preprocess_texts(texts):
    # You MUST implement this: convert list of strings to (input_ids, attention_mask)
    # that match your training data format
    input_ids = []
    attention_masks = []
    for text in texts:
        encoding = tokenizer.encode_plus(text,
                                         max_length=max_seq_length,
                                         truncation=True,
                                         padding='max_length',
                                         return_tensors='pt')
        input_ids.append(encoding['input_ids'])
        attention_masks.append(encoding['attention_mask'])
    
    input_ids = torch.cat(input_ids).to(device)
    attention_masks = torch.cat(attention_masks).to(device)
    return input_ids, attention_masks

# Define a prediction function compatible with LIME
def predict_fn(texts, batch_size=16):
    all_probs = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        input_ids, attention_mask = preprocess_texts(batch)
        with torch.no_grad():
            _, probs= model(input_ids, attention_mask)
            all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)


In [ ]:
# Replace with your class names
class_names = ['non-misogyny','misogyny']
explainer = LimeTextExplainer(class_names=class_names)

# Pick a test example
sample_text = "Women are already equal now — there's no need for feminism anymore"

# Explain the prediction
explanation = explainer.explain_instance(sample_text, predict_fn, num_features=10)
explanation.show_in_notebook(text=True)


In [ ]:
sample_text = "There aren’t many things that are more satisfying than telling a girl, 'No' "

# Explain the prediction
explanation = explainer.explain_instance(sample_text, predict_fn, num_features=10)
explanation.show_in_notebook(text=True)